In [ ]:
!pip install boto3 pyarrow --break-system-packages

In [ ]:
import sys
print(sys.executable)

In [3]:
import requests
import pandas as pd
import boto3
import io
import time

BASE = "https://ffiec.cfpb.gov/v2/data-browser-api/view"
BUCKET = "hmda-funnel-analytics-juanjo"

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

s3 = boto3.client("s3", region_name="us-east-2")

usecols_final = [
    "activity_year","action_taken","denial_reason-1","preapproval",
    "loan_amount","property_value","interest_rate","rate_spread",
    "discount_points","total_loan_costs","origination_charges",
    "derived_loan_product_type","derived_dwelling_category","conforming_loan_limit",
    "derived_ethnicity","derived_race","derived_sex","applicant_age","income",
    "debt_to_income_ratio","state_code","county_code",
    "ffiec_msa_md_median_family_income",
]
usecols_temp = ["occupancy_type", "business_or_commercial_purpose", "reverse_mortgage", "total_units"]

server_params = {
    "years": 2023,
    "loan_purposes": 1,
    "actions_taken": "1,2,3,4,5,7,8",
}

states_list = ["TX","FL","CA","NC","GA","IL","OH","NY","PA","MI",
               "VA","AZ","TN","NJ","IN","WA","MO","MD","MA","WI"]

def fetch_state(state_code):
    params = {**server_params, "states": state_code}
    kept_chunks = []
    with requests.get(f"{BASE}/csv", params=params, headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        for chunk in pd.read_csv(r.raw, usecols=usecols_final + usecols_temp,
                                  chunksize=100_000, low_memory=False):
            mask = (
                (chunk["occupancy_type"] == 1) &
                (chunk["business_or_commercial_purpose"] == 2) &
                (chunk["reverse_mortgage"] == 2) &
                (chunk["total_units"].astype(str) == "1")
            )
            kept_chunks.append(chunk.loc[mask, usecols_final])
    return pd.concat(kept_chunks, ignore_index=True) if kept_chunks else pd.DataFrame()

def upload_state_direct(state_code, max_retries=3):
    for attempt in range(1, max_retries + 1):
        try:
            df_st = fetch_state(state_code)
            buffer = io.BytesIO()
            df_st.to_parquet(buffer, index=False)
            buffer.seek(0)
            key = f"raw/hmda_{state_code}.parquet"
            s3.upload_fileobj(buffer, BUCKET, key)
            print(f"{state_code}: {df_st.shape[0]:,} filas subidas (intento {attempt})")
            return True
        except Exception as e:
            print(f"{state_code}: fallo en intento {attempt} -> {e}")
            time.sleep(5)
    print(f"{state_code}: NO se pudo subir después de {max_retries} intentos")
    return False

existing = {obj["Key"] for obj in s3.list_objects_v2(Bucket=BUCKET, Prefix="raw/").get("Contents", [])}

for st in states_list:
    key = f"raw/hmda_{st}.parquet"
    if key in existing:
        print(f"{st}: ya existe, salteado")
        continue
    upload_state_direct(st)
    time.sleep(1)

print("Listo.")

TX: 536,955 filas subidas (intento 1)
FL: 414,891 filas subidas (intento 1)
CA: fallo en intento 1 -> 'utf-8' codec can't decode byte 0x8b in position 1: invalid start byte
CA: fallo en intento 2 -> 'utf-8' codec can't decode byte 0x8b in position 1: invalid start byte
CA: 325,822 filas subidas (intento 3)
NC: fallo en intento 1 -> 'utf-8' codec can't decode byte 0x8b in position 1: invalid start byte
NC: 194,696 filas subidas (intento 2)
GA: fallo en intento 1 -> 'utf-8' codec can't decode byte 0x8b in position 1: invalid start byte
GA: 181,985 filas subidas (intento 2)
IL: fallo en intento 1 -> 'utf-8' codec can't decode byte 0x8b in position 1: invalid start byte
IL: 154,730 filas subidas (intento 2)
OH: fallo en intento 1 -> 'utf-8' codec can't decode byte 0x8b in position 1: invalid start byte
OH: 153,223 filas subidas (intento 2)
NY: fallo en intento 1 -> 'utf-8' codec can't decode byte 0x8b in position 1: invalid start byte
NY: 129,339 filas subidas (intento 2)
PA: fallo en inte